In [3]:
import wandb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

def setup_aaai_style():
    plt.rcParams.update({
        'font.size': 9,
        'axes.labelsize': 9,
        'legend.fontsize': 8,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'font.family': 'serif',
        'axes.grid': True,
        'axes.axisbelow': True,
        'grid.alpha': 0.3,
        'grid.linewidth': 0.8,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'axes.spines.top': False,
        'axes.spines.right': False,
        'axes.spines.left': True,
        'axes.spines.bottom': True,
        'axes.linewidth': 1.0,
        'xtick.direction': 'out',
        'ytick.direction': 'out',
        'lines.markersize': 4,
        'lines.linewidth': 1.2,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.titlepad': 4,
    })

def get_model_style():
    return {
        "GRU":    {"color": "#1f77b4", "marker": "s"},
        "RDDLGN": {"color": "#9467bd", "marker": "^"},
        "default":{"color": "#444444", "marker": "v"},
    }

def fetch_seq_length_data(sweep_id, model_name):
    api = wandb.Api()
    sweep = api.sweep(sweep_id)
    runs = sweep.runs
    data = []
    for run in runs:
        if run.state not in ["finished", "running"]:
            continue
        config = run.config
        summary = run.summary._json_dict if hasattr(run.summary, "_json_dict") else dict(run.summary)
        # Extract sequence length
        tokenizer_params = config.get('tokenizer', {}).get('params', {})
        seq_length = tokenizer_params.get('seq_length')
        # Extract train accuracy
        acc = None
        for k in ["train/metric/accuracy", "train_accuracy", "train/accuracy", "train/acc"]:
            if k in summary:
                acc = summary[k]
                break
        if seq_length is not None and acc is not None:
            data.append({
                "seq_length": int(seq_length),
                "accuracy": float(acc),
                "model": model_name,
                "run_id": run.id,
            })
    return pd.DataFrame(data)

def filter_min_accuracy(df, min_accuracy=0.20):
    if "accuracy" in df.columns:
        return df[df["accuracy"] >= min_accuracy].copy()
    return df

def select_xticks(seq_lengths, max_ticks=8):
    """Return at most max_ticks unique, evenly spaced x ticks."""
    seq_unique = np.unique(seq_lengths)
    if len(seq_unique) <= max_ticks:
        return seq_unique
    return np.linspace(seq_unique.min(), seq_unique.max(), max_ticks, dtype=int)

def plot_seq_length_vs_train_accuracy(gru_data, rddlgn_data, output_dir="seq_length_analysis_aaai"):
    setup_aaai_style()
    styles = get_model_style()
    fig, ax = plt.subplots(figsize=(3.25, 2.4), dpi=300)

    plotted = []
    for df, model in [(gru_data, "GRU"), (rddlgn_data, "RDDLGN")]:
        if df.empty: continue
        st = styles.get(model, styles["default"])
        x = np.array(df["seq_length"])
        y = np.array(df["accuracy"]) * 100
        scatter = ax.scatter(
            x, y,
            color=st["color"], marker=st["marker"],
            edgecolors="white", linewidths=0.9, alpha=0.8, label=model, s=28, zorder=3,
        )
        plotted.append(scatter)

    # X ticks: at most 8, all unique if ≤8, else linspace
    all_seq = pd.concat([gru_data["seq_length"], rddlgn_data["seq_length"]])
    if not all_seq.empty:
        xticks = select_xticks(all_seq, 8)
        ax.set_xticks(xticks)
        ax.set_xlim(all_seq.min() - 2, all_seq.max() + 2)

    ax.set_xlabel("Sequence Length", fontweight='bold', fontsize=9, labelpad=2)
    ax.set_ylabel("Train Accuracy (%)", fontweight='bold', fontsize=9, labelpad=2)
    ax.set_ylim(0, None)
    # Legend: only GRU and RDDLGN
    ax.legend(handles=plotted, frameon=False, ncol=1, loc="lower right")
    plt.tight_layout(pad=0.25)

    os.makedirs(output_dir, exist_ok=True)
    pdf_path = os.path.join(output_dir, "seq_length_vs_train_accuracy.pdf")
    fig.savefig(pdf_path, format="pdf", dpi=300, bbox_inches="tight", pad_inches=0.03)
    png_path = os.path.join(output_dir, "seq_length_vs_train_accuracy.png")
    fig.savefig(png_path, format="png", dpi=300, bbox_inches="tight", pad_inches=0.03)
    print(f"Plot saved as {pdf_path} and {png_path}")
    plt.close(fig)

def main():
    project = "sbuehrer-eth-z-rich/final_report"
    sweeps = {
        "prun4xui": "GRU",
        "9cbynw40": "RDDLGN",
    }

    dfs = {}
    for sweep_id, model_label in sweeps.items():
        sweep_ref = f"{project}/sweeps/{sweep_id}"
        print(f"Fetching sweep data for {model_label}: {sweep_ref}")
        df = fetch_seq_length_data(sweep_ref, model_label)
        if not df.empty:
            dfs[model_label] = filter_min_accuracy(df, min_accuracy=0.20)

    if not dfs:
        print("No data found for any sweep.")
        return

    os.makedirs("seq_length_analysis_aaai", exist_ok=True)
    for model, df in dfs.items():
        path = f"seq_length_analysis_aaai/{model.lower()}_seq_length_train_accuracy.csv"
        df.to_csv(path, index=False)
        print(f"{model} data saved to {path}")

    plot_seq_length_vs_train_accuracy(
        dfs.get("GRU", pd.DataFrame()),
        dfs.get("RDDLGN", pd.DataFrame()),
        output_dir="seq_length_analysis_aaai"
    )
    print("All outputs saved to seq_length_analysis_aaai/")

if __name__ == "__main__":
    main()

Fetching sweep data for GRU: sbuehrer-eth-z-rich/final_report/sweeps/prun4xui
Fetching sweep data for RDDLGN: sbuehrer-eth-z-rich/final_report/sweeps/9cbynw40
GRU data saved to seq_length_analysis_aaai/gru_seq_length_train_accuracy.csv
RDDLGN data saved to seq_length_analysis_aaai/rddlgn_seq_length_train_accuracy.csv
Plot saved as seq_length_analysis_aaai/seq_length_vs_train_accuracy.pdf and seq_length_analysis_aaai/seq_length_vs_train_accuracy.png
All outputs saved to seq_length_analysis_aaai/
